# 08 — Category Theory Extensions & Sheaf Neural Networks


> **Note.** The theory below is the foundation of the shipped `SchemaFunctor` (rename/merge/delete) and Kan-based migration in `cognition.cassette.migrate`. See **[15 — Schema Migration](cognition-workshop/15-schema-migration.md)** for the shipped tool and `store.migrate(functor, ...)` for the user-facing entrypoint. Sheaf coherence as a runtime scoring signal is still theory-only; H¹ discrepancy does live in the sheaf GNN (see **[12 — Sheaf GNN](12_sheaf_gnn.ipynb)**).

Four constructions applied to the cognition knowledge graph:

1. **Sheaf coherence** — NPMI presheaf on the anchor co-activation graph, replacing placeholder `coherence=0.0` with a real local-to-global consistency signal.
2. **Functorial data migration** — schema evolution via rename / merge / delete without re-ingestion.
3. **Left Kan extension** — discover "natural" anchor categories from raw SPLADE activations via spectral clustering.
4. **Sheaf neural network** — per-relation **restriction maps** `(P_fwd, P_bwd)` and a sheaf-Laplacian edge-discrepancy regularizer, wired into the GNN as a drop-in for R-GCN.

The first three are unsupervised signals you apply *to* the graph. The fourth *changes how the GNN propagates messages*.


## Setup: Geopolitical corpus and schema

In [ ]:
import json, numpy as np
from pathlib import Path

from cognition import (
    Cognition, CognitionConfig, AnchorSchema, Encoder,
    extract_infons, split_sentences,
    SheafCoherence, SchemaFunctor, FunctorialMigration, SchemaDiscovery,
)

# --- 30-anchor geopolitical schema ---
GEO_SCHEMA = {
    # Actors (12)
    "us":             {"type": "actor",    "tokens": ["us", "united states", "america", "washington"]},
    "china":          {"type": "actor",    "tokens": ["china", "chinese", "beijing"]},
    "russia":         {"type": "actor",    "tokens": ["russia", "russian", "moscow", "kremlin"]},
    "iran":           {"type": "actor",    "tokens": ["iran", "iranian", "tehran"]},
    "israel":         {"type": "actor",    "tokens": ["israel", "israeli", "jerusalem"]},
    "nato":           {"type": "actor",    "tokens": ["nato", "alliance", "atlantic"]},
    "eu":             {"type": "actor",    "tokens": ["eu", "european union", "brussels"]},
    "un":             {"type": "actor",    "tokens": ["un", "united nations", "security council"]},
    "india":          {"type": "actor",    "tokens": ["india", "indian", "delhi"]},
    "japan":          {"type": "actor",    "tokens": ["japan", "japanese", "tokyo"]},
    "palestine":      {"type": "actor",    "tokens": ["palestine", "palestinian", "gaza", "hamas"]},
    "african_union":  {"type": "actor",    "tokens": ["african union", "au", "african"]},
    # Relations (8)
    "sanction":       {"type": "relation", "tokens": ["sanction", "sanctions", "embargo", "restrict"]},
    "negotiate":      {"type": "relation", "tokens": ["negotiate", "negotiation", "talks", "diplomacy", "diplomatic"]},
    "deploy":         {"type": "relation", "tokens": ["deploy", "deployment", "send", "station", "troops"]},
    "attack":         {"type": "relation", "tokens": ["attack", "strike", "bomb", "assault", "offensive"]},
    "trade":          {"type": "relation", "tokens": ["trade", "export", "import", "tariff", "commerce"]},
    "invest":         {"type": "relation", "tokens": ["invest", "investment", "fund", "finance"]},
    "condemn":        {"type": "relation", "tokens": ["condemn", "denounce", "criticize", "oppose"]},
    "cooperate":      {"type": "relation", "tokens": ["cooperate", "cooperation", "collaborate", "joint", "partnership"]},
    # Features (6)
    "nuclear":        {"type": "feature",  "tokens": ["nuclear", "uranium", "enrichment", "atomic"]},
    "military":       {"type": "feature",  "tokens": ["military", "army", "defense", "weapon"]},
    "technology":     {"type": "feature",  "tokens": ["technology", "cyber", "ai", "digital"]},
    "territory":      {"type": "feature",  "tokens": ["territory", "border", "land", "sovereignty"]},
    "humanitarian":   {"type": "feature",  "tokens": ["humanitarian", "aid", "refugee", "crisis"]},
    "maritime":       {"type": "feature",  "tokens": ["maritime", "naval", "sea", "strait", "shipping"]},
    # Markets (4)
    "middle_east":    {"type": "market",   "tokens": ["middle east", "gulf", "levant"]},
    "east_asia":      {"type": "market",   "tokens": ["east asia", "pacific", "asia-pacific", "indo-pacific"]},
    "europe":         {"type": "market",   "tokens": ["europe", "european", "continent"]},
    "africa":         {"type": "market",   "tokens": ["africa", "sahel", "horn of africa", "sub-saharan"]},
}

# --- 24 geopolitical documents, 2004–2026 ---
GEO_DOCS = [
    {"id": "geo-001", "timestamp": "2004-08-02", "text": "African Union deployed peacekeeping troops to the Darfur region amid escalating humanitarian crisis and military attacks on civilian populations."},
    {"id": "geo-002", "timestamp": "2006-12-23", "text": "The UN Security Council imposed sanctions on Iran over its nuclear enrichment program, demanding suspension of uranium processing activities."},
    {"id": "geo-003", "timestamp": "2008-03-26", "text": "NATO expanded its military presence in Afghanistan while the African Union struggled to maintain peacekeeping operations in Somalia."},
    {"id": "geo-004", "timestamp": "2010-06-09", "text": "China and Japan engaged in maritime territorial disputes in the East China Sea, with both nations deploying naval vessels near contested islands."},
    {"id": "geo-005", "timestamp": "2012-11-14", "text": "Israel launched military operations in Gaza while the international community condemned the escalating violence and humanitarian crisis."},
    {"id": "geo-006", "timestamp": "2014-03-18", "text": "Russia annexed Crimea, prompting the EU and US to impose sweeping economic sanctions and NATO to deploy additional forces to Eastern Europe."},
    {"id": "geo-007", "timestamp": "2015-07-05", "text": "Iran nuclear deal signed after years of diplomatic negotiations between Iran and world powers, lifting trade sanctions in exchange for nuclear restrictions."},
    {"id": "geo-008", "timestamp": "2016-07-12", "text": "China rejected the international tribunal ruling on South China Sea territorial claims, deploying military assets to artificial islands in the disputed maritime region."},
    {"id": "geo-009", "timestamp": "2017-09-03", "text": "India and Japan announced a joint investment in infrastructure development across Southeast Asia to counter Chinese economic influence in the region."},
    {"id": "geo-010", "timestamp": "2018-06-12", "text": "US-China trade war escalated with new tariffs on technology exports, affecting global supply chains and East Asian markets."},
    {"id": "geo-011", "timestamp": "2019-10-09", "text": "Turkey launched military operations in northern Syria, drawing condemnation from the EU and complicating NATO alliance dynamics."},
    {"id": "geo-012", "timestamp": "2020-09-10", "text": "Israel signed the Abraham Accords, normalizing diplomatic relations with UAE and Bahrain in a historic Middle East peace agreement."},
    {"id": "geo-013", "timestamp": "2021-08-15", "text": "NATO forces withdrew from Afghanistan as the Taliban seized control, creating a humanitarian crisis and refugee emergency."},
    {"id": "geo-014", "timestamp": "2022-02-24", "text": "Russia invaded Ukraine, triggering the largest military conflict in Europe since WWII. NATO deployed rapid response forces and the EU imposed unprecedented economic sanctions."},
    {"id": "geo-015", "timestamp": "2022-08-02", "text": "US Speaker Pelosi visited Taiwan, escalating tensions between China and the US. China deployed military forces around Taiwan in response."},
    {"id": "geo-016", "timestamp": "2023-01-15", "text": "Japan announced a historic increase in military spending and cooperation with NATO, signaling a shift in East Asian security dynamics."},
    {"id": "geo-017", "timestamp": "2023-04-20", "text": "India emerged as a key diplomatic mediator, negotiating with both Russia and the West while expanding trade partnerships across Africa and the Middle East."},
    {"id": "geo-018", "timestamp": "2023-10-07", "text": "Hamas launched a major attack on Israel from Gaza, triggering Israeli military operations and international calls for humanitarian corridors."},
    {"id": "geo-019", "timestamp": "2024-02-14", "text": "EU imposed new sanctions on Iranian drone technology transfers to Russia, linking Middle East and European security concerns."},
    {"id": "geo-020", "timestamp": "2024-06-10", "text": "China and Russia conducted joint naval exercises in the Pacific, while Japan and the US strengthened their maritime defense cooperation."},
    {"id": "geo-021", "timestamp": "2024-09-01", "text": "African Union launched a continental technology investment initiative, partnering with India and Japan on digital infrastructure across Sub-Saharan Africa."},
    {"id": "geo-022", "timestamp": "2025-01-20", "text": "UN-mediated negotiations on the Ukraine conflict stalled as Russia rejected territorial concessions and NATO expanded its eastern border presence."},
    {"id": "geo-023", "timestamp": "2025-06-15", "text": "Iran announced a new nuclear cooperation agreement with China, drawing condemnation from Israel and the US while complicating Middle East diplomacy."},
    {"id": "geo-024", "timestamp": "2026-01-10", "text": "The EU and African Union signed a comprehensive trade and humanitarian partnership, investing in technology and infrastructure development."},
]

print(f"Schema: {len(GEO_SCHEMA)} anchors")
print(f"Corpus: {len(GEO_DOCS)} documents, {GEO_DOCS[0]['timestamp']} – {GEO_DOCS[-1]['timestamp']}")

In [ ]:
# Build schema + encoder + extract infons
schema = AnchorSchema(GEO_SCHEMA)
encoder = Encoder(schema=schema)
config = CognitionConfig(schema_path=None)

infons, edges = extract_infons(GEO_DOCS, encoder, schema, config)
print(f"Extracted {len(infons)} infons, {len(edges)} edges")
print(f"Unique triples: {len(set(inf.triple_key() for inf in infons))}")

---
## Part 1: Sheaf Coherence

The importance formula uses `coherence` as a weight, but extraction sets it to `0.0`.
A presheaf on the anchor co-activation graph fills this with a real signal:
do these three anchors genuinely co-occur across the corpus?

In [ ]:
# 1a. Encode all sentences and observe co-activations
sentences = []
for doc in GEO_DOCS:
    sentences.extend(split_sentences(doc["text"]))

activation_matrix = encoder.encode(sentences)
print(f"Activation matrix: {activation_matrix.shape}  (sentences x anchors)")

sheaf = SheafCoherence(schema.names)
sheaf.observe(activation_matrix, threshold=0.3)
sheaf.fit()

print(f"\nFiedler value (algebraic connectivity): {sheaf.fiedler_value:.4f}")
print(f"  > 0 means the anchor graph is connected (single global section)")

In [ ]:
# 1b. NPMI co-activation heatmap
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 10)
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
im = ax.imshow(sheaf.npmi, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
ax.set_xticks(range(sheaf.n))
ax.set_xticklabels(sheaf.anchor_names, rotation=90, fontsize=8)
ax.set_yticks(range(sheaf.n))
ax.set_yticklabels(sheaf.anchor_names, fontsize=8)
ax.set_title('Anchor NPMI Co-activation Matrix')
fig.colorbar(im, ax=ax, label='NPMI')
plt.tight_layout()
plt.show()

In [ ]:
# 1c. Score all infons
scores = sheaf.score_batch(infons)
for inf, s in zip(infons, scores):
    inf.coherence = s

print(f"Scored {len(infons)} infons")
print(f"Mean coherence: {np.mean(scores):.3f}")
print(f"Std coherence:  {np.std(scores):.3f}")

# Top and bottom
ranked = sorted(zip(infons, scores), key=lambda x: -x[1])
print("\nMost coherent triples:")
for inf, s in ranked[:5]:
    print(f"  {s:.3f}  <<{inf.predicate}, {inf.subject}, {inf.object}>>")

print("\nLeast coherent triples:")
for inf, s in ranked[-5:]:
    print(f"  {s:.3f}  <<{inf.predicate}, {inf.subject}, {inf.object}>>")

In [ ]:
# 1d. Anchor centrality — which anchors are hubs?
centrality = sheaf.anchor_centrality()
ranked_anchors = sorted(centrality.items(), key=lambda x: -x[1])

types = schema.types
print("Anchor centrality (top 15):")
for name, score in ranked_anchors[:15]:
    bar = '█' * int(score * 20)
    print(f"  {types.get(name, '?'):10s} {name:18s} {score:.3f} {bar}")

# Connected components
components = sheaf.component_structure()
print(f"\nConnected components: {len(components)}")
for i, comp in enumerate(components):
    print(f"  Component {i}: {len(comp)} anchors — {', '.join(sorted(comp)[:8])}{'...' if len(comp) > 8 else ''}")

---
## Part 2: Functorial Data Migration

Schema evolution without re-ingestion. A functor `F: Schema_old → Schema_new`
pushes forward all infons. Three operations: **rename**, **merge**, **delete**.

In [ ]:
# 2a. Rename: us → united_states, china → prc
rename_defs = dict(GEO_SCHEMA)
rename_defs["united_states"] = {"type": "actor", "tokens": ["us", "united states", "america", "washington"]}
rename_defs["prc"] = {"type": "actor", "tokens": ["china", "chinese", "beijing"]}
del rename_defs["us"]
del rename_defs["china"]
target_schema = AnchorSchema(rename_defs)

functor = SchemaFunctor(
    rename={"us": "united_states", "china": "prc"},
)
migration = FunctorialMigration(functor, schema, target_schema)
migrated_infons, migrated_edges = migration.migrate_all(infons, edges)

report = migration.report(infons, migrated_infons)
print("Rename migration report:")
for k, v in report.items():
    print(f"  {k}: {v}")

# Show some renamed triples
print("\nSample renamed triples:")
for inf in migrated_infons[:8]:
    print(f"  <<{inf.predicate}, {inf.subject}, {inf.object}>>  conf={inf.confidence:.3f}")

In [ ]:
# 2b. Merge: condemn + sanction → coerce
merge_defs = dict(GEO_SCHEMA)
merge_defs["coerce"] = {"type": "relation", "tokens": ["sanction", "condemn", "denounce", "restrict", "embargo"]}
del merge_defs["sanction"]
del merge_defs["condemn"]
merge_target = AnchorSchema(merge_defs)

merge_functor = SchemaFunctor(
    merge={"sanction": "coerce", "condemn": "coerce"},
)
merge_migration = FunctorialMigration(merge_functor, schema, merge_target)
merged_infons, merged_edges = merge_migration.migrate_all(infons, edges)

# Count coerce predicates
coerce_infons = [inf for inf in merged_infons if inf.predicate == "coerce"]
reinforced = [inf for inf in merged_infons if inf.reinforcement_count > 0]
print(f"Merge result: {len(infons)} → {len(merged_infons)} infons")
print(f"Infons with predicate='coerce': {len(coerce_infons)}")
print(f"Reinforced (merged duplicates): {len(reinforced)}")

print("\nCoerce triples:")
for inf in coerce_infons[:10]:
    print(f"  <<coerce, {inf.subject}, {inf.object}>>  reinforced={inf.reinforcement_count}  conf={inf.confidence:.3f}")

In [ ]:
# 2c. Delete: remove 'nuclear' — all infons mentioning nuclear are dropped
delete_functor = SchemaFunctor(delete={"nuclear"})
delete_defs = {k: v for k, v in GEO_SCHEMA.items() if k != "nuclear"}
delete_target = AnchorSchema(delete_defs)

delete_migration = FunctorialMigration(delete_functor, schema, delete_target)
deleted_infons, deleted_edges = delete_migration.migrate_all(infons, edges)

removed = len(infons) - len(deleted_infons)
print(f"Delete 'nuclear': {len(infons)} → {len(deleted_infons)} infons ({removed} removed)")
print(f"No infon should reference 'nuclear':")
nuclear_refs = [inf for inf in deleted_infons 
                if 'nuclear' in (inf.subject, inf.predicate, inf.object)]
print(f"  Nuclear references remaining: {len(nuclear_refs)}")

In [ ]:
# 2d. Composition: F2 ∘ F1 = F_composite
# F1: rename us → united_states
# F2: rename united_states → usa
# Composite: us → usa

f1 = SchemaFunctor(rename={"us": "united_states"})
f2 = SchemaFunctor(rename={"united_states": "usa"})
f_composite = SchemaFunctor(rename={"us": "usa"})

# Build schemas
s1_defs = dict(GEO_SCHEMA)
s1_defs["united_states"] = s1_defs.pop("us")
schema_mid = AnchorSchema(s1_defs)

s2_defs = dict(s1_defs)
s2_defs["usa"] = s2_defs.pop("united_states")
schema_final = AnchorSchema(s2_defs)

# Sequential: F1 then F2
m1 = FunctorialMigration(f1, schema, schema_mid)
step1_infons, step1_edges = m1.migrate_all(infons, edges)

m2 = FunctorialMigration(f2, schema_mid, schema_final)
step2_infons, step2_edges = m2.migrate_all(step1_infons, step1_edges)

# Direct: F_composite
sc_defs = dict(GEO_SCHEMA)
sc_defs["usa"] = sc_defs.pop("us")
schema_direct = AnchorSchema(sc_defs)

mc = FunctorialMigration(f_composite, schema, schema_direct)
direct_infons, direct_edges = mc.migrate_all(infons, edges)

# Compare
seq_triples = sorted(set(inf.triple_key() for inf in step2_infons))
dir_triples = sorted(set(inf.triple_key() for inf in direct_infons))

print(f"Sequential (F2 ∘ F1): {len(seq_triples)} unique triples")
print(f"Composite (F_comp):    {len(dir_triples)} unique triples")
print(f"Identical: {seq_triples == dir_triples} ✔" if seq_triples == dir_triples else f"MISMATCH ✘")

---
## Part 3: Schema-Free Discovery (Left Kan Extension)

No schema at all. SPLADE encodes raw text, spectral clustering on the
co-activation matrix discovers anchor categories from the data.

In [ ]:
# 3a. Discover anchors from raw text
sentences_raw = []
for doc in GEO_DOCS:
    sentences_raw.extend(split_sentences(doc["text"]))

discovery = SchemaDiscovery()
discovered_schema, discovered_anchors = discovery.discover(
    sentences_raw, n_anchors=15, min_doc_freq=2,
)

print(f"Discovered {len(discovered_anchors)} anchors from {len(sentences_raw)} sentences:\n")
for da in sorted(discovered_anchors, key=lambda x: -x.mean_activation):
    print(f"  {da.inferred_type:10s} {da.name:20s} tokens={da.tokens[:3]}  "
          f"size={da.size}  activation={da.mean_activation:.3f}  coherence={da.coherence:.1f}")

In [ ]:
# 3b. Use discovered schema to extract infons
disc_encoder = Encoder(schema=discovered_schema)
disc_infons, disc_edges = extract_infons(GEO_DOCS, disc_encoder, discovered_schema, config)

print(f"Discovered schema → {len(disc_infons)} infons")
print(f"Unique triples: {len(set(inf.triple_key() for inf in disc_infons))}")

# Type distribution
from collections import Counter
type_counts = Counter()
for name in discovered_schema.names:
    type_counts[discovered_schema.types[name]] += 1
print(f"\nType distribution: {dict(type_counts)}")

if disc_infons:
    print("\nSample discovered triples:")
    for inf in disc_infons[:10]:
        print(f"  <<{inf.predicate}, {inf.subject}, {inf.object}>>  conf={inf.confidence:.3f}")

---
## Part 4: Full Pipeline Integration

Combine all three: extract with sheaf coherence, query with persona valence,
walk temporal chains.

In [ ]:
# 4a. Full pipeline with sheaf-enhanced importance
from cognition import Cognition, CognitionConfig
import json, tempfile

# Write schema to temp file
schema_path = Path("data/geo_schema.json")
schema_path.parent.mkdir(exist_ok=True)
schema_path.write_text(json.dumps(GEO_SCHEMA, indent=2))

cog = Cognition(CognitionConfig(
    schema_path=str(schema_path),
    db_path="data/geo_category.db",
))

n = cog.ingest(GEO_DOCS, consolidate_now=True)
print(f"Ingested {n} infons")

# Apply sheaf coherence to stored infons
all_infons = cog.store.query_infons(limit=50000)
sentences = []
for doc in GEO_DOCS:
    sentences.extend(split_sentences(doc["text"]))

act_matrix = cog.encoder.encode(sentences)
sheaf = SheafCoherence(cog.schema.names)
sheaf.observe(act_matrix, threshold=0.3)
sheaf.fit()

for inf in all_infons:
    inf.coherence = sheaf.score_infon(inf)
cog.store.put_infons(all_infons)

print(f"Updated {len(all_infons)} infons with sheaf coherence")
print(f"Fiedler value: {sheaf.fiedler_value:.4f}")
print(f"Components: {len(sheaf.component_structure())}")

In [ ]:
# 4b. Query with persona valence + coherence
queries = [
    ("How has the conflict between Russia and NATO evolved?", "analyst"),
    ("What are the investment opportunities in East Asia?", "investor"),
    ("What nuclear threats exist in the Middle East?", "regulator"),
]

for query_text, persona in queries:
    result = cog.query(query_text, persona=persona, top_k=8)
    print(f"\n{'='*70}")
    print(f"Q: {query_text}")
    print(f"Persona: {result.persona} | Infons: {len(result.infons)} | Constraints: {len(result.constraints)}")
    
    for inf in result.infons[:5]:
        v = result.valence.get(inf.infon_id, 0)
        arrow = '▲' if v > 0.1 else '▼' if v < -0.1 else '─'
        print(f"  {arrow} <<{inf.predicate}, {inf.subject}, {inf.object}>>  "
              f"coh={inf.coherence:.3f} conf={inf.confidence:.3f}")
    
    if result.timeline:
        print(f"  Timeline: {result.timeline[0].timestamp} → {result.timeline[-1].timestamp} ({len(result.timeline)} events)")

In [ ]:
# 4c. Timeline visualization for top query
result = cog.query("How has the conflict between Russia and NATO evolved?", 
                   persona="analyst", top_k=50)

print(f"Timeline: {len(result.timeline)} events, {len(result.edges)} NEXT edges\n")

for inf in result.timeline:
    v = result.valence.get(inf.infon_id, 0)
    arrow = '▲' if v > 0.1 else '▼' if v < -0.1 else '─'
    bar = '█' * int(inf.coherence * 10)
    print(f"  {inf.timestamp}  {arrow} <<{inf.predicate}, {inf.subject}, {inf.object}>>  "
          f"coh={inf.coherence:.3f} {bar}")

In [ ]:
# 4d. Sheaf diagnostics summary
centrality = sheaf.anchor_centrality()
top_central = sorted(centrality.items(), key=lambda x: -x[1])[:10]

print("Sheaf Diagnostics")
print(f"  Fiedler value: {sheaf.fiedler_value:.4f}")
print(f"  Components: {len(sheaf.component_structure())}")
print(f"  Mean coherence: {np.mean([inf.coherence for inf in all_infons]):.3f}")
print(f"\n  Top central anchors (hubs connecting the graph):")
for name, score in top_central:
    bar = '█' * int(score * 20)
    print(f"    {schema.types.get(name, '?'):10s} {name:18s} {score:.3f} {bar}")

cog.close()
print("\nDone.")

---
## Part 5: Sheaf Neural Network layer

The first sheaf construction above (`SheafCoherence`) is an unsupervised
signal on the *anchor* graph. The sheaf *neural network* goes further:
it replaces each relation's single weight matrix `W_r` with a pair of
**restriction maps** `P_fwd[r], P_bwd[r]` that project each endpoint of
an edge into a shared edge stalk.

**The core idea (Bodnar et al., 2022 — Neural Sheaf Diffusion):**

- Every edge `(s -r-> t)` defines a stalk where both endpoints must
  agree for the relation's semantics to be coherent.
- `P_fwd[r]` projects the **source's** view of `r`.
- `P_bwd[r]` projects the **target's** view of `r`.
- The sheaf-Laplacian penalty `L_F = Σ ||P_fwd[r]·h_s − P_bwd[r]·h_t||²`
  is zero when all edges admit a consistent global section.

This is strictly more expressive than a single `W_r`: a relation can
apply one view when the source "speaks" into it and a different view
when the target "reads" from it. It also gives you an *unsupervised*
regularizer (`L_F`) that pushes embeddings toward structurally coherent
geometry without any labels.

Drop-in activation: `HypergraphReasoner(..., use_sheaf=True)`.


In [ ]:
from cognition.logic import SheafMessagePassingLayer, NUM_RELATIONS
import torch

# Instantiate the layer. It has per-relation P_fwd / P_bwd, initialized
# near identity so it starts roughly equal to an R-GCN baseline.
layer = SheafMessagePassingLayer(in_dim=32, out_dim=32)

print(f"relations:     {NUM_RELATIONS}")
print(f"P_fwd shape:   {layer.P_forward[0].shape}")
print(f"P_bwd shape:   {layer.P_backward[0].shape}")
print(f"params total:  {sum(p.numel() for p in layer.parameters()):,}")


In [ ]:
# The sheaf-Laplacian regularizer: measure and minimize edge
# discrepancy on a tiny 5-node synthetic graph. In isolation, training
# drives L_F → 0 (the sheaf admits a coherent global section).

torch.manual_seed(0)
h = torch.randn(5, 32)
edge_index = torch.tensor([[0, 1, 2, 3], [1, 2, 3, 4]], dtype=torch.long)
edge_types = torch.tensor([0, 1, 2, 3], dtype=torch.long)
edge_weights = torch.ones(4)

d0 = layer.sheaf_discrepancy(h, edge_index, edge_types, edge_weights).item()
print(f"L_F before training: {d0:.4f}")

opt = torch.optim.Adam(layer.parameters(), lr=1e-2)
for step in range(30):
    opt.zero_grad()
    d = layer.sheaf_discrepancy(h, edge_index, edge_types, edge_weights)
    d.backward()
    opt.step()

d1 = layer.sheaf_discrepancy(h, edge_index, edge_types, edge_weights).item()
print(f"L_F after 30 Adam steps: {d1:.4f}  ← monotone decrease")


### End-to-end: sheaf reasoner on the geopolitical corpus

Now wire it into a full `HypergraphReasoner`. The only change vs the
R-GCN baseline is the `use_sheaf=True` flag and the `laplacian_weight`
in `fit()`. Everything else — the IKL aggregators, the Dempster-Shafer
readout, the relevance-filtered combine — is unchanged.


In [ ]:
from cognition.logic import HypergraphReasoner

# cog is already built from Part 4 above (geopolitical corpus).
# If you're running this standalone, (re)build `cog` first.

reasoner_sheaf = HypergraphReasoner(
    cog.store, cog.encoder, cog.schema,
    hidden_dim=32, n_layers=2, use_sheaf=True,
)
graph = reasoner_sheaf.builder.build(feature_dim=32)
stats = reasoner_sheaf.fit(
    graph=graph, epochs=15,
    laplacian_weight=0.1, verbose=False,
)
print(f"sheaf-reasoner fit: epochs={stats['epochs']}, "
      f"final_loss={stats['final_loss']:.4f}")

r = reasoner_sheaf.reason("What is the state of Russia-NATO relations?")
print(f"verdict: {r.verdict}")
print(f"S={r.mass.supports:.2f}  R={r.mass.refutes:.2f}  "
      f"U={r.mass.uncertain:.2f}  θ={r.mass.theta:.2f}")


### When does the sheaf layer help?

On tiny, "clean" corpora the sheaf layer will *roughly match* the R-GCN
baseline — near-identity restriction maps don't yet have any asymmetry
to exploit, and with few conflicting edges there's nothing for the
Laplacian to regularize away.

The sheaf layer is expected to pay off on **larger, messier graphs
with conflicting relation views**: supply-chain networks where the same
`partners` edge means different things to two endpoints, climate FEVER
where a claim's "supports" from one paragraph contradicts "refutes"
from another, financial filings where counterparty-dependence is
asymmetric.

That's the next corpus — see §7 of `presentation/problem_formulation.html`.


---

## The Category-Theoretic View

| Category Theory | Cognition System |
|----------------|------------------|
| Objects | Anchors (typed vocabulary entries) |
| Morphisms | Infons (grounded triples connecting anchors) |
| Composition | NEXT chains (temporal sequencing) |
| Functor | Schema migration (structure-preserving map) |
| Presheaf | Sheaf coherence (local-to-global consistency) |
| Left Kan extension | Schema discovery (free construction from data) |
| Sheaf neural network | Per-relation `P_fwd` / `P_bwd` restriction maps + `L_F` regularizer |

| Natural transformation | Importance decay (systematic modification across all morphisms) |
| Colimit | Constraints (aggregation of co-supporting infons) |
| Pullback | Query results (intersection of anchor-filtered subgraphs) |

The knowledge graph is a category. Schemas are its sketches. Functors preserve structure across schema evolution. Sheaves measure consistency. And the Kan extension discovers structure from nothing.

**Previous:** [07 Cloud Deployment](07_cloud.ipynb)